<a href="https://colab.research.google.com/github/nafissadik2212/ML-BootCamp/blob/main/ML_BC_First_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#setting up the necessary libraries and cvs

import pandas as pd #to read files like xlms cvs for easier array conversion
import numpy as np #for mathematical calculations
#import matplotlib.pyplot as plt #for data visulazation
#followins are for data processing
from sklearn.model_selection import train_test_split #to train the machine
from sklearn.pipeline import Pipeline #to pielinr process
from sklearn.impute import SimpleImputer #to fill up missing values


from sklearn.preprocessing import StandardScaler, OneHotEncoder #for standardizing repectively mumarical and catagoy value
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression #linear model
from sklearn.ensemble import RandomForestRegressor #forest regressor model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
#importingthe file from the uplod file section form the path /content/
df = pd.read_csv('/content/sample_data/synthetic_noisy_dataset_40k_15col.csv')

#-------------------------------------------------------------------------------
print("Shape:", df.shape) #to see the dimensions of the table
df.head() #to see first 5 row of the table excluding the header row

Shape: (40200, 15)


,age,income,experience_years,education_years,hours_worked_per_week,satisfaction_score,department,city,performance_rating,projects_completed,training_hours,remote_work_ratio,tenure_years,overtime_hours,target_salary
0,58.0,69549.87,39.0,12.0,39.3,6.14,Engineering,Sylhet,2.0,5.0,35.9,0.497,39.0,2.8,68384.37
1,52.0,60693.22,32.0,16.0,29.5,7.23,Marketing,Dhaka,5.0,5.0,6.3,0.671,32.0,2.2,65594.91
2,43.0,45906.57,25.0,19.0,58.1,7.71,HR,Rajshahi,4.0,1.0,26.9,0.187,25.0,0.5,49246.79
3,58.0,65469.20,NaN,10.0,38.9,7.45,Engineering,Chittagong,4.0,3.0,13.3,0.255,39.0,13.7,72753.03
4,45.0,60345.96,24.0,17.0,34.6,4.05,Finance,Sylhet,4.0,8.0,7.1,0.612,22.0,0.0,NaN


In [ ]:
#---------------------------------------------------

#dropping row where target (targt_salary) value is missing
df = df.dropna(subset=['target_salary'])

#removing full row if it is repeated
df = df.drop_duplicates()

#removing space and amking all carcter lower case to standardize
df['department'] = df['department'].str.lower().str.strip()
df['city'] = df['city'].str.lower().str.strip()

#Using IQR handling outlier for numerical column
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.drop('target_salary')
#upper line will fetch data type float and int for column and will exclude the target-salary

#for the IQR function
for col in num_cols:
  Q1 = df[col].quantile(0.25) #to getthe cut off point for the lowest 25%
  Q3 =  df[col].quantile(0.75) #to get the cut off point for the top 25%
  IQR = Q3-Q1
  lower_bound = Q1 - 1.5*IQR #to get the lower boundary
  upper_bound = Q3 - 1.5*IQR #to get the upper boundary

  #cap the colmun for upper and lower boundaries to get the outliers vanished
  df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
  df[col] = np.where(df[col]<lower_bound, lower_bound, df[col])


In [ ]:
#selct the feature and target
x = df.drop(columns=['target_salary']) #for feature, remove the target
y = df['target_salary']


#train-test 80-20 split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)


#create prerocessing pipelines
cat_cols = ['department', 'city']
num_cols = x.select_dtypes(include=['float64', 'int64']).columns


#fill missing num data with median and then scale
num_transformer = Pipeline(steps=[

    ('imputer', SimpleImputer(strategy='median')),
     ('scaler', StandardScaler())

     ])
#above line will pipeline two step, one is it will fill the missing numerical vallue with median of SimpleImputer. two it will standarsize the value


#fill the missing catalgorical data and then converting into numerical fiormate
cat_transformer = Pipeline(steps=[
    ('imputr', SimpleImputer(strategy='most_frequent')),
    ('oneshot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
""" above line first filled the missing categories with mode
 then it took all those column and for each unique catregories it created a new column and as valu used 0 or 1 for that row, if that row had that category it will have 1 for that column.
 if there was any unknown it will handel the exception by puting 0s, and the last part will create NmunPy array for asy read
 """


#combine those array
preprocessor = ColumnTransformer(transformers=[('num', num_transformer, num_cols), ('cat', cat_transformer, cat_cols)])
#abovelne will take the previously modified array and will create an unified new array with new column names


#initialize modle to train the new machine
#linearRegresson model loaded
lr_pipeline = Pipeline(steps=[
      ('preprocessor', preprocessor),
      ('model', LinearRegression())
      ])
#RandomForest model loaded
rf_pipeline = Pipeline(steps=[

    ('preprocessor', preprocessor),
     ('model', RandomForestRegressor(n_estimators=50, random_state=42))

    ])

#now the cream part --> train the model
lr_pipeline.fit(x_train, y_train)
rf_pipeline.fit(x_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'income', 'experience_years', 'education_years',
       'hours_worked_per_week', 'satisfaction_score', 'performance_rating',
       'projects_completed', 'training_hours', 'remote_work_ratio',
       'tenure_years', 'overtime_hours'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputr',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('oneshot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['department', 'city'])])),
                ('model',
                 RandomForestRegressor(n_estimators=50, random_state=42))])

In [ ]:
#predicting time
lr_preds = lr_pipeline.predict(x_test)
rf_preds = rf_pipeline.predict(x_test)

def evalute_model(name, y_true, predictions):
  mae = mean_absolute_error(y_true, predictions)
  rmse = np.sqrt(mean_squared_error(y_true, predictions))
  r2 = r2_score(y_true, predictions)

  print("\n---",name,"Performance ---\n")
  print("MAE is :", mae)
  print("RMSE is:", rmse)
  print("R2 is  :", r2)


evalute_model("Linear Regression", y_test, lr_preds)
evalute_model("Random Forest Regressor", y_test, rf_preds)




--- Linear Regression Performance ---

MAE is : 13698.867776605563
RMSE is: 18754.455361733897
R2 is  : 0.09312879802892082

--- Random Forest Regressor Performance ---

MAE is : 13504.567150360814
RMSE is: 19000.322746557664
R2 is  : 0.06919511462401462
